# ⚖️ Full Pipeline (tối ưu + resume) — `results.json` 2000 câu

Self-contained, GPU Colab. **Phase A nhanh (~10 phút)** + **lưu `retrieved.json`** → nếu Colab ngắt,
chạy lại KHÔNG mất việc (Phase A bỏ qua, Phase B resume từ checkpoint).

**Trước khi chạy:** `Runtime → T4 GPU`. Upload `corpus_articles.jsonl` + `stage1_questions.json`.
(Resume phiên mới: upload thêm `retrieved.json` và/hoặc `results.json` đã tải về trước đó.)

## 1. Cài thư viện

In [ ]:
!pip install -q "sentence-transformers>=3.0" "transformers>=4.44" accelerate bitsandbytes rank_bm25 tiktoken sentencepiece

## 2. Kiểm tra GPU

In [ ]:
!nvidia-smi -L

## 3. Upload file
`corpus_articles.jsonl` + `stage1_questions.json` (+ `retrieved.json`/`results.json` nếu resume).

In [ ]:
from google.colab import files
up = files.upload()
import json
questions = json.load(open("stage1_questions.json", encoding="utf-8"))
print("questions:", len(questions), "| files:", list(up.keys()))

## 4. Phase A — Retrieval (tối ưu: fp16 reranker, CAND=20, batch-embed). Lưu `retrieved.json`, bỏ qua nếu đã có.

In [ ]:
import json, re, os, numpy as np, torch
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from rank_bm25 import BM25Okapi

EMBED_ID, RERANK_ID = "AITeamVN/Vietnamese_Embedding", "AITeamVN/Vietnamese_Reranker"
MAX_SEQ, RERANK_MAX, CAND, TOP_K, MIN_SCORE, MARGIN, RR_BATCH = 1024, 256, 20, 8, 0.0, 4.0, 32
DEV = "cuda" if torch.cuda.is_available() else "cpu"
RET_PATH = "retrieved.json"
def tok(t): return re.findall(r"\w+", (t or "").lower(), flags=re.UNICODE)

if os.path.exists(RET_PATH):
    cache = {int(k): v for k, v in json.load(open(RET_PATH, encoding="utf-8")).items()}
    print(f"Đã có retrieved.json ({len(cache)} câu) → BỎ QUA Phase A.")
else:
    corpus = [json.loads(l) for l in open("corpus_articles.jsonl", encoding="utf-8")
              if l.strip() and json.loads(l).get("doc_number")]
    docs_text = [f"{r['title']}\n{r['text']}" for r in corpus]
    print("corpus:", len(corpus), "| device:", DEV)

    emb = SentenceTransformer(EMBED_ID, device=DEV); emb.max_seq_length = MAX_SEQ
    if os.path.exists("corpus_emb.npy"):                       # đã upload embeddings → khỏi embed lại
        corpus_emb = np.load("corpus_emb.npy").astype("float32")
        assert len(corpus_emb) == len(corpus), "corpus_emb.npy không khớp corpus — xóa file để embed lại"
        print("Dùng corpus_emb.npy đã upload → BỎ QUA embed corpus")
    else:
        corpus_emb = emb.encode(docs_text, batch_size=128, normalize_embeddings=True,
                                convert_to_numpy=True, show_progress_bar=True).astype("float32")
        np.save("corpus_emb.npy", corpus_emb)
        print("Đã lưu corpus_emb.npy (tải về ở cell 4b để lần sau khỏi embed)")
    q_emb = emb.encode([q["question"] for q in questions], batch_size=128, normalize_embeddings=True,
                       convert_to_numpy=True, show_progress_bar=True).astype("float32")
    bm25 = BM25Okapi([tok(t) for t in docs_text])
    rr_tok = AutoTokenizer.from_pretrained(RERANK_ID)
    rr = AutoModelForSequenceClassification.from_pretrained(RERANK_ID).to(DEV).eval()
    if DEV == "cuda": rr = rr.half()

    @torch.no_grad()
    def rerank(query, idxs):
        pairs = [[query, docs_text[i]] for i in idxs]; out = []
        for j in range(0, len(pairs), RR_BATCH):
            enc = rr_tok(pairs[j:j+RR_BATCH], padding=True, truncation=True,
                         max_length=RERANK_MAX, return_tensors="pt").to(DEV)
            out.extend(rr(**enc).logits.view(-1).float().cpu().tolist())
        return out

    cache = {}
    for n, q in enumerate(questions):
        dense = np.argsort(corpus_emb @ q_emb[n])[::-1][:CAND]
        bm = bm25.get_scores(tok(q["question"]))
        sp = [i for i in np.argsort(bm)[::-1][:CAND] if bm[i] > 0]
        rrf = {}
        for rank, i in enumerate(dense): rrf[int(i)] = rrf.get(int(i), 0) + 1/(60+rank+1)
        for rank, i in enumerate(sp):    rrf[int(i)] = rrf.get(int(i), 0) + 1/(60+rank+1)
        fused = sorted(rrf, key=rrf.get, reverse=True)[:CAND]
        scored = sorted(zip(fused, rerank(q["question"], fused)), key=lambda x: x[1], reverse=True)
        seen, dd = set(), []
        for i, s in scored:
            k = (corpus[i]["doc_number"], corpus[i]["article"])
            if k not in seen: seen.add(k); dd.append((i, s))
        sel = []
        if dd:
            sel = [dd[0]]; top = dd[0][1]
            for i, s in dd[1:TOP_K]:
                if s < MIN_SCORE or s < top - MARGIN: break
                sel.append((i, s))
        cache[int(q["id"])] = [{k: corpus[i][k] for k in ("doc_number","clean_name","article","title","text")} for i, _ in sel]
        if (n+1) % 200 == 0: print(f"retrieved {n+1}/{len(questions)}")

    json.dump({str(k): v for k, v in cache.items()}, open(RET_PATH, "w", encoding="utf-8"), ensure_ascii=False)
    print("Đã lưu retrieved.json. Giải phóng VRAM...")
    del emb, rr, corpus_emb, q_emb; torch.cuda.empty_cache()

## 4b. (Khuyến nghị) Tải `retrieved.json` về để phòng Colab ngắt
Lần sau upload lại file này ở cell 3 → khỏi chạy lại Phase A.

In [ ]:
from google.colab import files
import os
for f in ("retrieved.json", "corpus_emb.npy"):
    if os.path.exists(f): files.download(f)

## 5. Phase B1a — Nạp LLM (CHẠY 1 LẦN; đừng chạy lại để khỏi nạp lại model)

In [ ]:
import json, re, os, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Nạp cache retrieval từ file (resume-safe, không phụ thuộc RAM của Phase A)
cache = {int(k): v for k, v in json.load(open("retrieved.json", encoding="utf-8")).items()}
questions = json.load(open("stage1_questions.json", encoding="utf-8"))
print("cache:", len(cache), "| questions:", len(questions))

# Giải phóng VRAM Phase A (embed + reranker) trước khi nạp LLM — tránh CUDA OOM
import gc
for _v in ["emb", "rr", "rr_tok", "corpus_emb", "q_emb", "bm25"]:
    globals().pop(_v, None)
gc.collect(); torch.cuda.empty_cache()
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# 3B nhẹ, chắc chắn vừa T4 16GB. Nếu còn dư VRAM, đổi sang "Qwen/Qwen2.5-7B-Instruct".
LLM_ID = "Qwen/Qwen2.5-3B-Instruct"
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
ltok = AutoTokenizer.from_pretrained(LLM_ID)
llm = AutoModelForCausalLM.from_pretrained(LLM_ID, quantization_config=bnb,
                                           device_map="auto", low_cpu_mem_usage=True)
print("LLM loaded:", LLM_ID, "| VRAM free check OK")

## 5b. Phase B1b — Helpers + TEST 5 câu (chạy lại thoải mái, KHÔNG nạp lại model)

In [ ]:
def build_prompt(query, ctx):
    blocks = [f"[{i}] {d.get('clean_name','')} ({d.get('doc_number','')}) — {d.get('article','')}\n{d.get('text','')}"
              for i, d in enumerate(ctx, 1)]
    context_str = "\n\n".join(blocks) if blocks else "(Không có căn cứ.)"
    must = ", ".join(sorted({d.get("article", "") for d in ctx if d.get("article")}))
    system = ("Bạn là trợ lý pháp lý AI cho doanh nghiệp SME Việt Nam. Trả lời DỰA HOÀN TOÀN trên CƠ SỞ PHÁP LÝ; "
              "tuyệt đối không bịa.\nTốt = chính xác (số liệu/điều kiện/thời hạn lấy từ điều luật), đầy đủ, thực tiễn, "
              "rõ ràng dễ hiểu.\nBẮT BUỘC nêu rõ số Điều + tên văn bản cho mỗi căn cứ. "
              "Nếu không đủ căn cứ, nói rõ + khuyến nghị tham vấn luật sư.")
    user = f"CƠ SỞ PHÁP LÝ:\n{context_str}\n\nCÂU HỎI: {query}\n\nTrích dẫn rõ các điều: {must}.\nCâu trả lời:"
    return system, user

@torch.no_grad()
def gen(system, user, max_new=350):
    # apply_chat_template(tokenize=False) → string, rồi tokenizer → inputs dict (chuẩn Qwen,
    # tránh lỗi BatchEncoding.shape ở vài bản transformers).
    text = ltok.apply_chat_template([{"role": "system", "content": system}, {"role": "user", "content": user}],
                                    add_generation_prompt=True, tokenize=False)
    inputs = ltok([text], return_tensors="pt").to(llm.device)
    out = llm.generate(**inputs, max_new_tokens=max_new, do_sample=False, pad_token_id=ltok.eos_token_id)
    gen_ids = out[0][inputs.input_ids.shape[1]:]
    return ltok.decode(gen_ids, skip_special_tokens=True).strip()

def build_fields(ctx):
    docs, arts = [], []
    for d in ctx:
        c, nm, a = d.get("doc_number", ""), d.get("clean_name", ""), d.get("article", "")
        if not c or not a: continue
        if f"{c}|{nm}" not in docs: docs.append(f"{c}|{nm}")
        if f"{c}|{nm}|{a}" not in arts: arts.append(f"{c}|{nm}|{a}")
    return docs, arts

def ensure_cit(ans, ctx):
    present = set(re.findall(r"Điều\s+(\d+)", ans))
    miss = {}
    for d in ctx:
        m = re.match(r"Điều\s+(\d+)", d.get("article", ""))
        if m and m.group(1) not in present:
            miss.setdefault(d.get("clean_name", ""), []).append(d["article"])
    if not miss: return ans
    return ans.rstrip() + "\n\nCăn cứ pháp lý áp dụng: " + "; ".join(f"{', '.join(a)} ({nm})" for nm, a in miss.items()) + "."

for q in questions[:5]:
    ctx = cache.get(int(q["id"]), [])
    _, ra = build_fields(ctx)
    ans = ensure_cit(gen(*build_prompt(q["question"], ctx)), ctx) if ctx else "Chưa tìm thấy căn cứ pháp lý phù hợp."
    print("=" * 90); print("Q%s: %s" % (q["id"], q["question"])); print("Điều:", ra); print("ANSWER:\n" + ans)
print("\n>>> Ổn thì chạy Phase B2.")

## 6. Phase B2 — FULL 2000 câu (checkpoint mỗi 50, resume). ~1-2h.

In [ ]:
results = []
if os.path.exists("results.json"):
    try: results = json.load(open("results.json", encoding="utf-8"))
    except Exception: results = []
done = {r["id"] for r in results}
print(f"resume: đã có {len(done)} câu")

for q in questions:
    qid = int(q["id"])
    if qid in done: continue
    ctx = cache.get(qid, [])
    rd, ra = build_fields(ctx)
    if ctx:
        try: ans = ensure_cit(gen(*build_prompt(q["question"], ctx)), ctx)
        except Exception: ans = "Căn cứ pháp lý liên quan: " + "; ".join(ra)
    else:
        ans = "Chưa tìm thấy căn cứ pháp lý phù hợp. Khuyến nghị tham vấn luật sư."
    results.append({"id": qid, "question": q["question"], "answer": ans, "relevant_docs": rd, "relevant_articles": ra})
    if len(results) % 50 == 0:
        json.dump(sorted(results, key=lambda r: r["id"]), open("results.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
        print(f"checkpoint {len(results)}/{len(questions)}")

results.sort(key=lambda r: r["id"])
json.dump(results, open("results.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print("DONE:", len(results))

## 7. Đóng gói + tải `submission.zip`

In [ ]:
import zipfile, json
assert len(json.load(open("results.json"))) == len(json.load(open("stage1_questions.json"))), "Thiếu câu!"
with zipfile.ZipFile("submission.zip", "w", zipfile.ZIP_DEFLATED) as z:
    z.write("results.json", arcname="results.json")
from google.colab import files
files.download("submission.zip")

## 2 cách chạy

**Cách A — Tất cả trên Colab:** chạy cell 1→7 → tải `submission.zip`. (LLM 3B trên GPU, ~1-2h.)

**Cách B — Nặng ở Colab, LLM ở local** (đúng "tải về chạy phần còn lại"):
1. Colab: chạy cell **1→4** (embed + retrieve, ~10p) → **cell 4b tải `retrieved.json`** (+ `corpus_emb.npy` để tái dùng). DỪNG ở đây (bỏ qua Phase B).
2. Local: bỏ `retrieved.json` vào `data/`, rồi:
   ```bash
   ollama serve &
   ollama pull qwen2.5:7b-instruct-q4_K_M
   python scratch/generate_submission.py --retrieved data/retrieved.json --questions data/stage1_questions.json
   ```
   → Ollama (Qwen-**7B**) sinh answer ở local (~2h, checkpoint mỗi 50, ngắt → chạy lại tự resume) → `results.json` + `submission.zip`.

→ Cách B: phần GPU-nặng (embed + rerank) ở Colab; LLM chạy local không lo giới hạn session Colab, lại dùng được Qwen-7B (chất lượng hơn 3B).

---

## Resume khi Colab ngắt (Cách A)
1. Trước đó đã tải `retrieved.json` (cell 4b) và `results.json` (checkpoint — tải thủ công nếu cần).
2. Phiên mới: chạy cell 1-3, **upload `corpus_articles.jsonl`, `stage1_questions.json`, `retrieved.json`, `results.json`**.
3. Chạy cell 4 (tự bỏ qua vì có retrieved.json) → cell 5 → cell 6 (resume từ checkpoint).

## Tốc độ
- Phase A ~10 phút (fp16 reranker, CAND=20, max_len=256, batch-embed query).
- Phase B (LLM) ~1-2h cho 2000 câu — checkpoint mỗi 50, ngắt thì chạy lại cell 6 sẽ tiếp.